In [12]:
import clickhouse_connect

In [13]:
import pandas as pd
import requests

API_KEY = "67ad1bea200726.95055451"
BASE_URL = "https://eodhd.com/api"



In [14]:
CLICKHOUSE_HOST = '54.234.38.203'
CLICKHOUSE_PORT = 8123
CLICKHOUSE_USER = 'chain8'
CLICKHOUSE_PASS = 'c8_2025'
CLICKHOUSE_DB = 'default'
TABLE_NAME = 'tsx_eod'

# ------------------------
# ClickHouse: Connect
# ------------------------
client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASS,
    database=CLICKHOUSE_DB
)

## List all supported exchanges

In [15]:
url = f"{BASE_URL}/exchanges-list/?api_token={API_KEY}&fmt=json"
resp = requests.get(url)
df_exchanges = pd.DataFrame(resp.json())
df_exchanges.head()
df_exchanges.to_csv("exchanges_list.csv", index=False)

In [16]:
df_exchanges.head(76)

,Name,Code,OperatingMIC,Country,Currency,CountryISO2,CountryISO3
0,USA Stocks,US,"XNAS, XNYS, OTCM",USA,USD,US,USA
1,London Exchange,LSE,XLON,UK,GBP,GB,GBR
2,NEO Exchange,NEO,NEOE,Canada,CAD,CA,CAN
3,TSX Venture Exchange,V,XTSX,Canada,CAD,CA,CAN
4,Toronto Exchange,TO,XTSE,Canada,CAD,CA,CAN
...,...,...,...,...,...,...,...
71,Government Bonds,GBOND,None,Unknown,Unknown,,
72,Money Market Virtual Exchange,MONEY,None,Unknown,Unknown,,
73,Europe Fund Virtual Exchange,EUFUND,None,Unknown,EUR,,
74,Istanbul Stock Exchange,IS,XIST,Turkey,TRY,TR,TUR


In [17]:
currencies = ['INR', 'CAD','USD', 'EUR']
df_filtered = df_exchanges[df_exchanges['Currency'].isin(currencies)]
exchange_codes = df_filtered['Code'].tolist()
print(f"Total Exchanges Matching Criteria: {len(exchange_codes)}")
print(exchange_codes)

Total Exchanges Matching Criteria: 27
['US', 'NEO', 'V', 'TO', 'BE', 'HM', 'XETRA', 'DU', 'F', 'MU', 'HA', 'STU', 'LU', 'VI', 'PA', 'BR', 'MC', 'LS', 'AS', 'HE', 'IR', 'NSE', 'AT', 'IL', 'ZSE', 'EUFUND', 'CC']


In [20]:
from time import sleep

all_tickers = []

for exch in exchange_codes:
    try:
        url = f"{BASE_URL}/exchange-symbol-list/{exch}?api_token={API_KEY}&fmt=json"
        resp = requests.get(url)

        if resp.status_code == 200:
            tickers = resp.json()
            if tickers:
                df = pd.DataFrame(tickers)
                df['Exchange'] = exch
                all_tickers.append(df)
                print(f"✅ Fetched {len(df)} tickers from {exch}")
            else:
                print(f"⚠️ No tickers found for {exch}")
        else:
            print(f"❌ Failed for {exch}, Status Code: {resp.status_code}")

        sleep(1)  # Be nice to API (rate limiting)
    except Exception as e:
        print(f"❌ Error fetching {exch}: {e}")

# ------------------------
# Step 4: Combine and Save
# ------------------------
if all_tickers:
    df_all_tickers = pd.concat(all_tickers, ignore_index=True)
    df_all_tickers.to_csv("all_tickers_metadata.csv", index=False)
    print("✅ Saved to all_tickers_metadata.csv")
else:
    print("❌ No tickers fetched!")

✅ Fetched 51434 tickers from US
✅ Fetched 295 tickers from NEO
✅ Fetched 1481 tickers from V
✅ Fetched 2567 tickers from TO
✅ Fetched 3220 tickers from BE
✅ Fetched 611 tickers from HM
✅ Fetched 3061 tickers from XETRA
✅ Fetched 1442 tickers from DU
✅ Fetched 11865 tickers from F
✅ Fetched 2312 tickers from MU
✅ Fetched 218 tickers from HA
✅ Fetched 4826 tickers from STU
✅ Fetched 5 tickers from LU
✅ Fetched 80 tickers from VI
✅ Fetched 1202 tickers from PA
✅ Fetched 117 tickers from BR
✅ Fetched 274 tickers from MC
✅ Fetched 36 tickers from LS
✅ Fetched 466 tickers from AS
✅ Fetched 191 tickers from HE
✅ Fetched 59 tickers from IR
✅ Fetched 2657 tickers from NSE
✅ Fetched 146 tickers from AT
✅ Fetched 118 tickers from IL
✅ Fetched 20 tickers from ZSE
✅ Fetched 70971 tickers from EUFUND
✅ Fetched 2290 tickers from CC
✅ Saved to all_tickers_metadata.csv


In [30]:
df_tm = pd.read_csv("all_tickers_metadata.csv")

In [31]:
df_tm.head()

,Code,Name,Country,Exchange,Currency,Type,Isin
0,0P000070L2,RBC $U.S. Money Market Fund A,USA,US,USD,FUND,NaN
1,0P0000A2WI,Fidelity American High Yield Sr B,USA,US,USD,FUND,NaN
2,0P0000A412,Franklin U.S. Opportunities Fund A,USA,US,USD,FUND,NaN
3,0P0000O6WE,Nh Unique College Investing Plan Fidelity 500 ...,USA,US,USD,FUND,NaN
4,0P0000O753,Oh Collegeadvantage 529 Vanguard Wellington Op...,USA,US,USD,FUND,NaN


In [32]:
df_tm.shape

(161974, 7)

In [42]:
df_ind = df_tm[df_tm['Currency']=='INR']

In [43]:
df_ind['Type'].unique()

array(['FUND', 'Common Stock', 'ETF'], dtype=object)

In [33]:
df_tm['Type'].unique()

array(['FUND', 'Common Stock', 'ETF', 'Mutual Fund', 'Notes',
       'Preferred Stock', 'Unit', 'BOND', 'ETC', 'Note', nan,
       'DE000A2GS609', 'CH0019304531', 'Certificate', 'INDEX',
       'INE438K01021', 'DE0005322218', 'Currency'], dtype=object)

In [35]:
df_tm[df_tm['Type']=='INDEX']

,Code,Name,Country,Exchange,Currency,Type,Isin
75032,V4F3,VDAX 3M,Unknown,F,EUR,INDEX,NaN
80217,DE000SL0BRJ7,Solactive Developed Markets Healthcare 150 Ind...,Germany,STU,USD,INDEX,DE000SL0BRJ7
80218,DE000SL0CP60,Solactive GBS Developed Markets ex United Stat...,Germany,STU,USD,INDEX,DE000SL0CP60
80861,HARVESTN,PR ALT HARV NTR,Germany,STU,USD,INDEX,NaN
80862,HARVESTP,PR ALT HARV PR INDEX,Germany,STU,USD,INDEX,XC0006013624
83775,BIOTK,NEXT BIOTECH,Netherlands,PA,EUR,INDEX,NaN
83818,CACLG,CAC Large 60,France,PA,EUR,INDEX,NaN
83819,CACSH,CAC 40 SHORT,Netherlands,PA,EUR,INDEX,NaN
83844,CESGP,CAC 40 ESG,Netherlands,PA,EUR,INDEX,NaN
83999,ENPME,PEA-PME 150,Netherlands,PA,EUR,INDEX,NaN


In [44]:
# Filter for Common Stock and Index
df_filtered = df_tm[df_tm['Type'].isin(['Common Stock', 'INDEX'])]

# Check the result
print(df_filtered.shape)
df_filtered.head()

(49639, 7)


,Code,Name,Country,Exchange,Currency,Type,Isin
12,A,Agilent Technologies Inc,USA,US,USD,Common Stock,US00846U1016
13,AA,Alcoa Corp,USA,US,USD,Common Stock,US0138721065
31,AABB,Asia Broadband Inc,USA,US,USD,Common Stock,US04518L1008
44,AABVF,Aberdeen International Inc,USA,US,USD,Common Stock,NaN
48,AACAF,AAC Technologies Holdings Inc,USA,US,USD,Common Stock,NaN


In [45]:
df_filtered.to_csv('tickers_metadata_commonstock_index.csv', index=False)

In [46]:
df_filtered['Type'].unique()


array(['Common Stock', 'INDEX'], dtype=object)

In [47]:
import pandas as pd
from clickhouse_connect import get_client

# ------------------------
# ClickHouse Connection
# ------------------------
client = get_client(
    host='54.234.38.203',
    port=8123,
    username='chain8',
    password='c8_2025',
    database='default'
)

# ------------------------
# Load Filtered Metadata
# ------------------------
# Update filename as per your file
df = pd.read_csv("tickers_metadata_commonstock_index.csv")

# Clean up columns to avoid insert errors
columns = ['Code', 'Name', 'Country', 'Exchange', 'Currency', 'Type', 'Isin']
for col in columns:
    df[col] = df[col].fillna('').astype(str)

print(f"✅ Loaded dataframe with shape: {df.shape}")

# ------------------------
# Drop Table if Exists
# ------------------------
client.command("DROP TABLE IF EXISTS tickers_metadata")

# ------------------------
# Create Table in ClickHouse
# ------------------------
client.command("""
CREATE TABLE tickers_metadata (
    Code String,
    Name String,
    Country String,
    Exchange String,
    Currency String,
    Type String,
    Isin String,
    created_on DateTime DEFAULT now()
) ENGINE = MergeTree()
ORDER BY (Code)
""")
print("✅ ClickHouse table 'tickers_metadata' created.")

# ------------------------
# Insert Data
# ------------------------
client.insert_df("tickers_metadata", df)
print("✅ Data inserted into 'tickers_metadata' successfully.")


✅ Loaded dataframe with shape: (49639, 7)
✅ ClickHouse table 'tickers_metadata' created.
✅ Data inserted into 'tickers_metadata' successfully.


## List all tickers on TSX exchange

In [9]:
# List all tickers on TSX
url = f"{BASE_URL}/exchange-symbol-list/TO?api_token={API_KEY}&fmt=json"
resp = requests.get(url)

# Print response status and preview
print("Status Code:", resp.status_code)
print("Preview:\n", resp.text[:300])

# Convert to DataFrame
tickers = resp.json()
df_tickers = pd.DataFrame(tickers)

# Show some sample tickers
print(df_tickers.head())

# Save to CSV
df_tickers.to_csv("tsx_tickers_list.csv", index=False)

Status Code: 200
Preview:
 [{"Code":"0P0000704A","Name":"RBC North American Value Fund A","Country":"Canada","Exchange":"TO","Currency":"CAD","Type":"FUND","Isin":null},{"Code":"0P00007060","Name":"RBC Canadian Money Market Fund A","Country":"Canada","Exchange":"TO","Currency":"CAD","Type":"FUND","Isin":null},{"Code":"0P00007
         Code                                      Name Country Exchange  \
0  0P0000704A           RBC North American Value Fund A  Canada       TO   
1  0P00007060          RBC Canadian Money Market Fund A  Canada       TO   
2  0P00007061                RBC Canadian Equity Fund A  Canada       TO   
3  0P00007065                  RBC mondial d'Ã©nergie A  Canada       TO   
4  0P00007069  RBC Portefeuille de croissance sÃ©lect A  Canada       TO   

  Currency  Type  Isin  
0      CAD  FUND  None  
1      CAD  FUND  None  
2      CAD  FUND  None  
3      CAD  FUND  None  
4      CAD  FUND  None  


In [10]:
df = pd.read_csv("tsx_tickers_list.csv")

In [11]:
df.head()

,Code,Name,Country,Exchange,Currency,Type,Isin
0,0P0000704A,RBC North American Value Fund A,Canada,TO,CAD,FUND,NaN
1,0P00007060,RBC Canadian Money Market Fund A,Canada,TO,CAD,FUND,NaN
2,0P00007061,RBC Canadian Equity Fund A,Canada,TO,CAD,FUND,NaN
3,0P00007065,RBC mondial d'Ã©nergie A,Canada,TO,CAD,FUND,NaN
4,0P00007069,RBC Portefeuille de croissance sÃ©lect A,Canada,TO,CAD,FUND,NaN


In [ ]:
df.shape

In [ ]:
df['Type'].unique()

In [ ]:
df[df['Type']=='Preferred Stock']

In [ ]:
df[df['Type']=='Common Stock']

In [ ]:
print(df.columns.tolist())


In [ ]:
import pandas as pd
from clickhouse_connect import get_client

# Load CSV
df = pd.read_csv("tsx_tickers_list.csv")

# Clean up columns — ensure all are string type to avoid insert errors
for col in ['Code', 'Name', 'Country', 'Exchange', 'Currency', 'Type', 'Isin']:
    df[col] = df[col].fillna('').astype(str)


# Drop table if already exists
client.command("DROP TABLE IF EXISTS tickers_metadata")

# Create table with ClickHouse
client.command("""
CREATE TABLE tickers_metadata (
    Code String,
    Name String,
    Country String,
    Exchange String,
    Currency String,
    Type String,
    Isin String,
    created_on DateTime DEFAULT now()
) ENGINE = MergeTree()
ORDER BY (Code)
""")

# Insert DataFrame
client.insert_df("tickers_metadata", df)

print("✅ Tickers metadata inserted into ClickHouse.")


 ## EOD OHLCV for AAPL.US (can get only for last one year)

In [ ]:
symbol = "SHOP.TO"  # Format: {ticker}.{exchange_code}
BASE_URL = "https://eodhd.com/api"

url = f"{BASE_URL}/eod/{symbol}?api_token={API_KEY}&fmt=json"
resp = requests.get(url)

# Preview response
print("Status Code:", resp.status_code)
print("Preview:\n", resp.text[:300])

# Convert to DataFrame
data = resp.json()
df = pd.DataFrame(data)
df.to_csv("shopify_tsx_eod.csv", index=False)
df.head()

In [ ]:
# Step 1: Convert 'date' column to datetime if not already
df['date'] = pd.to_datetime(df['date'])

# Step 2: Get the earliest date
min_date = df['date'].min()

print("Earliest date:", min_date)


In [ ]:
# Step 1: Convert 'date' column to datetime if not already
df['date'] = pd.to_datetime(df['date'])

# Step 2: Get the earliest date
max_date = df['date'].max()

print("latest date:", max_date)


## Bulk EOD OHLCV for all US tickers (latest day)

In [ ]:
url = f"{BASE_URL}/eod-bulk-last-day/TO?api_token={API_KEY}&fmt=json"

resp = requests.get(url)
print(resp.status_code, resp.headers.get('Content-Type'))
print(resp.text[:300])

data = resp.json()
df = pd.DataFrame(data)
df.to_csv("canada_tsx_eod.csv", index=False)

In [ ]:
df = pd.read_csv("canada_tsx_eod.csv")

In [ ]:
df.head()

In [ ]:
df.shape

## Splits data for AAPL.US

In [ ]:
url = f"{BASE_URL}/splits/AAPL.US?api_token={API_KEY}&fmt=json"
resp = requests.get(url)

# Check if the response is valid JSON
try:
    data = resp.json()
    df_splits = pd.DataFrame(data)
    display(df_splits.head())
except ValueError:
    print("Failed to decode JSON")
    print("Status Code:", resp.status_code)
    print("Response Text Preview:\n", resp.text[:500])

## Dividends data for AAPL.US


In [ ]:
url = f"{BASE_URL}/dividends/AAPL.US?api_token={API_KEY}&fmt=json"
resp = requests.get(url)
# Check if the response is valid JSON
try:
    data = resp.json()
    df_dividends = pd.DataFrame(data)
    display(df_dividends.head())
except ValueError:
    print("Failed to decode JSON")
    print("Status Code:", resp.status_code)
    print("Response Text Preview:\n", resp.text[:500])


## Exchange trading details (US)



In [ ]:
url = f"{BASE_URL}/exchange-details/US?api_token={API_KEY}&fmt=json"
resp = requests.get(url)
try:
    data = resp.json()
    df_details = pd.DataFrame(data)
    display(df_details.head())
except ValueError:
    print("Failed to decode JSON")
    print("Status Code:", resp.status_code)
    print("Response Text Preview:\n", resp.text[:500])
